In [1]:
import cv2
import numpy as np
import pandas as pd
import os, glob
from pathlib import Path

In [2]:
BASE_DIR   = r"/home/hasan/coding/MoneyLens/ai/Dataset_ocr"
ARRAYS_DIR = os.path.join(BASE_DIR, "preprocessed")
SPLITS     = ["train", "valid", "test"]

IMG_H, IMG_W = 32, 128
CHANNELS = 1

PRIORITAS = {
    "total_transaksi"  : "KRITIS",
    "tanggal"          : "KRITIS",
    "total_harga_barang": "PENTING",
    "harga_satuan"     : "PENTING",
    "QTY"              : "TAMBAHAN",
    "nama_produk"      : "TAMBAHAN",
}

print("=" * 60)
print("EVALUASI PREPROCESSING OCR (BERSIH)")
print("=" * 60)

EVALUASI PREPROCESSING OCR (BERSIH)


In [3]:
def contrast(arr):
    return float(arr.std())

def sharpness(arr):
    img = (arr[:,:,0] * 255).astype(np.uint8)
    return float(cv2.Laplacian(img, cv2.CV_64F).var())

def black_ratio(arr):
    return float((arr < 0.5).mean())

def edge_density(arr):
    img = (arr[:,:,0] * 255).astype(np.uint8)
    edges = cv2.Canny(img, 50, 150)
    return float(edges.mean() / 255)

In [4]:
results = []

for split in SPLITS:
    arrays_dir = os.path.join(ARRAYS_DIR, split, "arrays")

    if not os.path.exists(arrays_dir):
        print(f"[{split}] folder tidak ditemukan")
        continue

    npy_files = glob.glob(os.path.join(arrays_dir, "*.npy"))

    if not npy_files:
        print(f"[{split}] tidak ada file npy")
        continue

    for npy_path in npy_files:
        fname = Path(npy_path).stem

        # ambil kelas dari nama file
        cls = None
        for c in PRIORITAS:
            if c in fname:
                cls = c
                break

        if cls is None:
            continue

        try:
            arr = np.load(npy_path)

            if arr.shape != (IMG_H, IMG_W, CHANNELS):
                continue

            results.append({
                "kelas": cls,
                "contrast": contrast(arr),
                "sharp": sharpness(arr),
                "black_ratio": black_ratio(arr),
                "edge": edge_density(arr),
            })

        except Exception as e:
            print(f"[ERROR] {fname}: {e}")

In [5]:
df = pd.DataFrame(results)

if df.empty:
    print("\n[WARNING] Tidak ada data terbaca")
    exit()

In [6]:
print(f"\n{'Kelas':<22} {'N':>4} {'Contrast':>10} {'Sharp':>10} {'Black':>8} {'Edge':>8}")
print("-"*70)

for cls in PRIORITAS:
    rows = df[df["kelas"] == cls]

    if rows.empty:
        print(f"{cls:<22} {'0':>4}")
        continue

    print(f"{cls:<22} "
          f"{len(rows):>4} "
          f"{rows['contrast'].mean():>10.4f} "
          f"{rows['sharp'].mean():>10.2f} "
          f"{rows['black_ratio'].mean():>8.3f} "
          f"{rows['edge'].mean():>8.3f}")


Kelas                     N   Contrast      Sharp    Black     Edge
----------------------------------------------------------------------
total_transaksi         401     0.1596     317.85    0.085    0.111
tanggal                 383     0.1385     755.88    0.061    0.152
total_harga_barang     1027     0.1379     193.96    0.055    0.101
harga_satuan            739     0.1371     184.71    0.062    0.100
QTY                     950     0.1212      74.28    0.050    0.054
nama_produk            1031     0.1444    3498.10    0.061    0.168


In [7]:
print("\n" + "="*60)
print("ANALISIS KELAS KRITIS")
print("="*60)

for cls in ["total_transaksi", "tanggal"]:
    rows = df[df["kelas"] == cls]

    if rows.empty:
        print(f"\n{cls}: tidak ada data")
        continue

    sharp = rows["sharp"].mean()
    black = rows["black_ratio"].mean()

    print(f"\n{cls}:")
    print(f"  Sharpness : {sharp:.2f}")
    print(f"  BlackRatio: {black:.3f}")

    if sharp < 40:
        print("  MASALAH: blur → crop salah / bbox melenceng")

    if black < 0.05:
        print("  MASALAH: teks hilang (crop terlalu sempit)")

    if black > 0.6:
        print("  MASALAH: terlalu banyak background/noise")


ANALISIS KELAS KRITIS

total_transaksi:
  Sharpness : 317.85
  BlackRatio: 0.085

tanggal:
  Sharpness : 755.88
  BlackRatio: 0.061


In [8]:
out_csv = os.path.join(BASE_DIR, "evaluasi_preprocessing.csv")
df.to_csv(out_csv, index=False)

print(f"\nHasil disimpan ke: {out_csv}")


Hasil disimpan ke: /home/hasan/coding/MoneyLens/ai/Dataset_ocr/evaluasi_preprocessing.csv
